# Enhanced S3 to COG Converter with Automatic AWS Authentication

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Support for multiple AWS authentication methods**

Author: Kyle Lesinger (Enhanced version)

In [4]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm
import re

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [5]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked,
)

from convert_utilities_improved import (
  convert_to_proper_CRS_and_cogify_improved,
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


## Monitoring Memory Usage

# Useful links

[drcs_activations OLD Directory](https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/)

[VEDA docs for file naming conventions](https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html)

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [6]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [7]:
EVENT_NAME = '202504_SevereWx_US'
#old name
#under drcs_activations
PRODUCT_NAME = 'sentinel2'


PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [8]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

# For very large files (>10GB)
large_file_config = {
  "default_chunk_size": 256,       # Small chunks
  "memory_limit_mb": 250,          # Conservative memory limit
  "aggressive_gc": True,
  "single_band_mode": True,        # Process bands one at a time
  "use_streaming": True,           # Stream to avoid download
  "cleanup_immediate": True
}

## Initialize AWS S3 Client with automatic credential detection

In [9]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name='nasa-disasters', verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name='nasa-disasters', verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, 'nasa-disasters', PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 55 .tif files in the S3 bucket.


['drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2A_MNDWI_20250408_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2A_NDVI_20250322_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2A_NDVI_20250408_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2A_trueColor_20250322_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2A_trueColor_20250408_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2B_NDVI_20250322_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2B_trueColor_20250322_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2C_MNDWI_20250409_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2C_NDVI_20250409_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/JAN_S2C_trueColor_20250409_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/LZK_S2B_MNDWI_20250407_merged.tif',
 'drcs_activations/202504_SevereWx_US/sentinel2/LZK_S2B_NDVI_202

In [10]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

In [11]:
# For simplicity, let's use python list comprehension to return the files
# We may need to rename them in different ways for different products
# We will do a similar process later

## NOTE --- We can actually use these objects since they have the same path as the s3 files. We will call them again later

ndvi = [f for f in keys if "NDVI" in f]#NDVI
true = [f for f in keys if "true" in f]#true
mndwi = [f for f in keys if "MNDWI" in f]#MNDWI

## Configure bucket and paths (no need to create session manually)

In [12]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [13]:
def convert_date(date_str):
    """
    Convert to YYYY-MM-DD.
    
    Args:
        datetime_str: String like '20250731'
    
    Returns:
        String like '2025-07-31'
    """
    # Extract components
    year = date_str[0:4]
    month = date_str[4:6]
    day = date_str[6:8]
    
    # Format with dashes and colons, add Z for UTC
    return f"{year}-{month}-{day}"

# Test
date_str = '20250731'
result = convert_date(date_str)
print(result)

2025-07-31


In [14]:
def create_cog_filename(f, EVENT_NAME):
    """Create COG filename for NDVI files."""
    # Extract directory, filename, and extension
    directory, filename = os.path.split(f)
    stem, ext = os.path.splitext(filename)

    # Find all 8-digit date patterns
    dates = re.findall(r"\d{8}", stem)

    # Remove dates from the stem
    stem_clean = re.sub(r"_?\d{8}", "", stem)

    # Build new stem
    cog_filename = f"{EVENT_NAME}_{stem_clean}_{convert_date(dates[0])}_day.tif"
    return cog_filename


filter_str = 'NDVI'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:
  202504_SevereWx_US_JAN_S2A_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_JAN_S2A_NDVI_merged_2025-04-08_day.tif
  202504_SevereWx_US_JAN_S2B_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_JAN_S2C_NDVI_merged_2025-04-09_day.tif
  202504_SevereWx_US_LZK_S2B_NDVI_merged_2025-04-07_day.tif
  202504_SevereWx_US_LZK_S2C_NDVI_merged_2015-03-13_day.tif
  202504_SevereWx_US_LZK_S2C_NDVI_merged_2025-04-09_day.tif
  202504_SevereWx_US_MEG_S2A_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_MEG_S2A_NDVI_merged_2025-04-08_day.tif
  202504_SevereWx_US_MEG_S2B_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_MEG_S2C_NDVI_merged_2025-04-09_day.tif
  202504_SevereWx_US_OHX_S2A_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_OHX_S2A_NDVI_merged_2025-04-08_day.tif
  202504_SevereWx_US_OHX_S2B_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_PAH_S2A_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_PAH_S2A_NDVI_merged_2025-04-08_day.tif
  202504_SevereWx_U

There's three types of NDVI files in this list, so I will handle renaming all three with conditional statements:

In [ ]:
# # Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename, 
                                target_dir = "Sentinel-2/NDVI", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202504_SevereWx_US_JAN_S2A_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_JAN_S2A_NDVI_merged_2025-04-08_day.tif
  202504_SevereWx_US_JAN_S2B_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_JAN_S2C_NDVI_merged_2025-04-09_day.tif
  202504_SevereWx_US_LZK_S2B_NDVI_merged_2025-04-07_day.tif
  202504_SevereWx_US_LZK_S2C_NDVI_merged_2015-03-13_day.tif
  202504_SevereWx_US_LZK_S2C_NDVI_merged_2025-04-09_day.tif
  202504_SevereWx_US_MEG_S2A_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_MEG_S2A_NDVI_merged_2025-04-08_day.tif
  202504_SevereWx_US_MEG_S2B_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_MEG_S2C_NDVI_merged_2025-04-09_day.tif
  202504_SevereWx_US_OHX_S2A_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_OHX_S2A_NDVI_merged_2025-04-08_day.tif
  202504_SevereWx_US_OHX_S2B_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_PAH_S2A_NDVI_merged_2025-03-22_day.tif
  202504_SevereWx_US_PAH_S2A_NDVI_merged_2025-04-08_day.tif
  202504_SevereWx_US_

Band 1:   3%|▎         | 64/1989 [00:04<02:23, 13.41chunks/s]


   [MEMORY] High usage: 730.6 MB, forcing cleanup...


Band 1:   4%|▎         | 73/1989 [00:05<03:30,  9.10chunks/s]


   [MEMORY] High usage: 733.9 MB, forcing cleanup...


...

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=-0.268550306558609, max=0.7002775073051453, center sample non-zero=1000000/1000000
            Estimated data coverage: 80.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpzkau8wwg_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpenmkdb14.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-2/NDVI/202504_SevereWx_US_OHX_S2B_NDVI_merged_2025-03-22_day.tif


In [ ]:
# Check current cache status using the imported function
check_cache_status()

stop

# Process RGB files

In [ ]:
# Process RGB files with chunked processing
if ndvi:
    print("\n" + "="*50)
    print("🎨 Processing RGB Files (Chunked)")
    print("="*50)
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    
    # Use the chunked conversion function
    def chunked_converter(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, cog_filename, cog_data_bucket, cog_data_prefix, 
            local_output_dir, chunk_config=CHUNK_CONFIG
        )
    
    rgb_results = process_file_batch(
        file_list=ndvi,
        s3_client=s3_client,
        config=config_ndvi,
        filename_creator_func=create_cog_filename,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True
    )
    all_files_processed = pd.concat([all_files_processed, rgb_results], ignore_index=True)
    
    # Print overall summary
    print_batch_summary(all_files_processed)

In [ ]:
# Process RGB files with chunked processing
if mndwi:
    print("\n" + "="*50)
    print("🎨 Processing RGB Files (Chunked)")
    print("="*50)
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    
    # Use the chunked conversion function
    def chunked_converter(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, cog_filename, cog_data_bucket, cog_data_prefix, 
            local_output_dir, chunk_config=CHUNK_CONFIG
        )
    
    rgb_results = process_file_batch(
        file_list=mndwi,
        s3_client=s3_client,
        config=config_mndwi,
        filename_creator_func=create_cog_filename,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True
    )
    all_files_processed = pd.concat([all_files_processed, rgb_results], ignore_index=True)
    
    # Print overall summary
    print_batch_summary(all_files_processed)

In [ ]:
# Process RGB files with chunked processing
if true:
    print("\n" + "="*50)
    print("🎨 Processing RGB Files (Chunked)")
    print("="*50)
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    
    # Use the chunked conversion function
    def chunked_converter(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, cog_filename, cog_data_bucket, cog_data_prefix, 
            local_output_dir, chunk_config=CHUNK_CONFIG
        )
    
    rgb_results = process_file_batch(
        file_list=true,
        s3_client=s3_client,
        config=config_true,
        filename_creator_func=create_cog_filename,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True
    )
    all_files_processed = pd.concat([all_files_processed, rgb_results], ignore_index=True)
    
    # Print overall summary
    print_batch_summary(all_files_processed)

In [ ]:
# Display final results
print(f"\n📊 Final Processing Results:")
print(f"Total files processed: {len(all_files_processed)}")
print(f"\nProcessed files DataFrame:")
all_files_processed

## Check STATUS
[Disasters Bucket](https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/)